## Preamble and sampling

In [120]:
import nltk
import polars as pl
import numpy as np

from collections import Counter
from datasets import load_dataset
from nltk.tokenize import word_tokenize

In [107]:
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [108]:
load_dataset("roneneldan/TinyStories", split="train").to_parquet("tinystories.parquet")
load_dataset("SimpleStories/SimpleStories", split="train").to_parquet("simplestories.parquet")

Creating parquet from Arrow format: 100%|██████████| 32/32 [01:06<00:00,  2.07s/ba]


3142783327

In [110]:
tinystories_df = pl.scan_parquet("tinystories.parquet")
# tinystories_df = pl.scan_parquet("tinystories.parquet").head(50000).collect()

In [111]:
simplestories_df = pl.scan_parquet("simplestories.parquet")
# simplestories_df = pl.scan_parquet("simplestories.parquet").head(50000).collect()

## Length comparison

In [112]:
tinystories_df = tinystories_df.with_columns(
    pl.col("text").str.split(" ").list.len().alias("word_count")
)

In [113]:
%%time
"""
flesch-kincaid approx function in pure polars
"""
tinystories_df = tinystories_df.with_columns([
    # Count words
    pl.col("text").str.split(" ").list.len().alias("word_count"),
    # Count sentences (split on . ! ?)
    pl.col("text").str.count_matches(r"[.!?]+").alias("sentence_count"),
    # Approximate syllables: count vowel groups per word
    pl.col("text").str.count_matches(r"[aeiouAEIOU]+").alias("syllable_count"),
]).with_columns(
    (
        0.39 * (pl.col("word_count") / pl.col("sentence_count").clip(lower_bound=1))
        + 11.8 * (pl.col("syllable_count") / pl.col("word_count").clip(lower_bound=1))
        - 15.59
    ).alias("flesch_kincaid_score")
)

simplestories_df = simplestories_df.with_columns([
    # Count words
    pl.col("story").str.split(" ").list.len().alias("word_count"),
    # Count sentences (split on . ! ?)
    pl.col("story").str.count_matches(r"[.!?]+").alias("sentence_count"),
    # Approximate syllables: count vowel groups per word
    pl.col("story").str.count_matches(r"[aeiouAEIOU]+").alias("syllable_count"),
]).with_columns(
    (
        0.39 * (pl.col("word_count") / pl.col("sentence_count").clip(lower_bound=1))
        + 11.8 * (pl.col("syllable_count") / pl.col("word_count").clip(lower_bound=1))
        - 15.59
    ).alias("flesch_kincaid_score")
)

CPU times: user 0 ns, sys: 462 μs, total: 462 μs
Wall time: 1.09 ms


In [114]:
ts_result = tinystories_df.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_score").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_score").std(),
    
)

ss_result = simplestories_df.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_score").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_score").std(),
    
)

In [115]:
%%time
print("Tiny stories")
ts_result.collect()

Tiny stories
CPU times: user 1min 10s, sys: 3.08 s, total: 1min 13s
Wall time: 53.9 s


word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
171.832831,77.416249,4.108448,1.513775


In [116]:
%%time
print("Simple stories")
ss_result.collect()

Simple stories
CPU times: user 1min 36s, sys: 1.92 s, total: 1min 38s
Wall time: 1min 14s


word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
222.87192,102.668153,4.936471,1.348981


## Compression ratio and Self-BLEU homogenization score

In [119]:
# random subsample from each 
from diversity import (
    compression_ratio,
    homogenization_score,
    ngram_diversity_score,
)
# hs = homogenization_score(texts, method='self-bleu')

[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [125]:
tinystories_df.head(1000).collect()

text,word_count,sentence_count,syllable_count,flesch_kincaid_score
str,u32,u32,u32,f64
"""One day, a little girl named L…",132,10,177,5.380727
"""Once upon a time, there was a …",140,14,181,3.565714
"""One day, a little fish named F…",166,17,203,2.648356
"""Once upon a time, in a land fu…",161,14,209,4.213012
"""Once upon a time, there was a …",127,11,170,4.708003
…,…,…,…,…
"""Tom and Lily were playing in t…",349,58,452,2.039246
"""Sara and Ben are friends. They…",170,38,220,1.425325
"""Lily liked to write with her c…",310,36,388,2.537366


In [143]:
def lazy_sample(lf, sample_size=1000, seed=42):

    return lf.with_row_index().filter(
        (pl.col("index").hash(seed=seed) % sample_size) == 0
    ).drop("index").collect()

In [ ]:
%%time
lazy_sample(tinystories_df)